# INFORM localization smoke test

In [9]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
print("Current folder:", repo_root)
print("Files:", [p.name for p in repo_root.iterdir() if p.is_dir() or p.suffix in [".toml", ".md", ".txt"]])
sys.path.insert(0, r"C:\Users\laura\OneDrive\Documenti\GitHub\INFORM")

Current folder: c:\Users\laura\OneDrive\Documenti\GitHub\INFORM\tests
Files: ['.ipynb_checkpoints']


## 1. Test import

In [10]:
from nerve_model.experiment import Experiment
from nerve_model.fiber_population import MotorFiberPopulation
from nerve_model.nerve_section import CircularFascicleTopography
from nerve_model.implant import Implant
from nerve_model.recruitment_curves import RecruitmentCurves

from localization.candidate_generation import create_loc_candidates
from localization.bayesian_localization import performLocalizationCluster, localize_functional_cluster

print("All imports OK")

All imports OK


## 2. Test candidate grid

In [11]:
candidates_grid = create_loc_candidates(
    nerve_radius=1.5,
    limCandidateStd=(0.15, 0.5),
    limCandidateNum=(200, 200),
    nTriesLocs=(20, 20),
    nTriesStd=20,
    nTriesNum=1,
)

print("Candidate grid shape:", candidates_grid.shape)
print("First 5 candidates:")
print(candidates_grid[:5])

assert candidates_grid.shape[1] == 4
assert candidates_grid.shape[0] > 0
print("Candidate generation OK")

Candidate grid shape: (5520, 4)
First 5 candidates:
[[ -0.55263158  -1.34210526   0.15       200.        ]
 [ -0.55263158  -1.34210526   0.16842105 200.        ]
 [ -0.55263158  -1.34210526   0.18684211 200.        ]
 [ -0.55263158  -1.34210526   0.20526316 200.        ]
 [ -0.55263158  -1.34210526   0.22368421 200.        ]]
Candidate generation OK


## 3. Test `RecruitmentCurves`

In [12]:
import numpy as np

n_sites = 4
n_groups = 2
n_steps = 20

amplitudes = np.linspace(0, 1, n_steps)

# Fake monotonic recruitment curves: shape (n_sites, n_groups, n_steps)
recruitment_values = np.zeros((n_sites, n_groups, n_steps))

for site in range(n_sites):
    for group in range(n_groups):
        threshold = 0.2 + 0.1 * group + 0.03 * site
        recruitment_values[site, group, :] = 1 / (1 + np.exp(-20 * (amplitudes - threshold)))

rc = RecruitmentCurves(
    recruitment_values=recruitment_values,
    amplitudes=amplitudes,
)

i50 = rc.compute_i50()

print("Recruitment values shape:", rc.recruitment_values.shape)
print("I50 shape:", i50.shape)
print(i50)

assert i50.shape == (n_sites, n_groups)
print("RecruitmentCurves OK")

Recruitment values shape: (4, 2, 20)
I50 shape: (4, 2)
[[0.19954587 0.29960372]
 [0.23028555 0.33042803]
 [0.25976387 0.35956706]
 [0.28997647 0.39020505]]
RecruitmentCurves OK


## 4. Mini-test localization 

In [20]:
from sklearn.gaussian_process.kernels import Matern
from sklearn.preprocessing import StandardScaler

class FakeActivationPredictor:
    def predict(self, X):
        return np.zeros(X.shape[0])


class FakeFiberPopulation:
    def __init__(self):
        self.n_fibers = 10
        self.n_groups = 1
        self.cluster_ids = np.zeros(self.n_fibers, dtype=int)
        self.cluster_locs = np.array([[0.0, 0.0]])
        self.cluster_std = np.array([0.25])
        self.cluster_num = np.array([10])
        self.fem_node_lims = np.array([[0, 1]] * self.n_fibers)
        self.n_fem_nodes = self.n_fibers
        self.n_nodes = 1
        self.fem_node_fiber_ids = np.arange(self.n_fibers)
        self.node_ids = np.ones((self.n_fibers, 1), dtype=int)
        self.diameters = np.ones(self.n_fibers)
        self.locs = np.zeros((self.n_fibers, 2))


class FakeNerveTopography:
    nerve_radius = 1.5
    fascicles = None


class FakeImplant:
    n_sites = 4
    site_locs = np.zeros((4, 2))


class FakeExperiment:
    def __init__(self, cluster_locs=None, cluster_std=None, cluster_num=None):
        self.fiber_population = FakeFiberPopulation()
        self.nerve_topography = FakeNerveTopography()
        self.implant = FakeImplant()
        self.activation_predictor = FakeActivationPredictor()
        self.lead_field_matrix = np.zeros((10, 4))

        if cluster_locs is None:
            self.candidate_centers = np.array([[0.0, 0.0]])
        else:
            arr = np.atleast_2d(np.asarray(cluster_locs, dtype=float))
            # accetta sia (2,) sia (n_candidates, 2)
            if arr.shape[1] != 2:
                arr = arr.reshape(-1, 2)
            self.candidate_centers = arr

    @classmethod
    def from_existing_experiment(
        cls,
        experiment,
        fiber_population=None,
        has_struct_info=False,
        cluster_locs=None,
        cluster_std=None,
        cluster_num=None,
    ):
        return cls(cluster_locs=cluster_locs, cluster_std=cluster_std, cluster_num=cluster_num), np.arange(10)

    def load_lead_field_matrix(self, *args, **kwargs):
        return self.lead_field_matrix

    def generate_recruitment_curves(self, amp_lims, n_steps, method="from_self"):
        target_center = np.array([0.3, -0.2])
        amplitudes = np.linspace(amp_lims[0], amp_lims[1], n_steps)
        base_curve = 1 / (1 + np.exp(-15 * (amplitudes - 0.5)))

        n_candidates = self.candidate_centers.shape[0]
        values = np.zeros((self.implant.n_sites, n_candidates, n_steps))

        for c in range(n_candidates):
            distance = np.linalg.norm(self.candidate_centers[c] - target_center)
            penalty = np.exp(-distance)
            curve = np.clip(base_curve * (0.2 + 0.8 * penalty), 0.0, 1.0)
            for site in range(self.implant.n_sites):
                values[site, c, :] = curve

        if not np.all(np.isfinite(values)):
            print("NaN/inf nelle curve!")

        return RecruitmentCurves(values, amplitudes)


# Monkey-patch Experiment used inside bayesian_localization for this software-only test.
import localization.bayesian_localization as bl
import localization.localization_utils_reference as _ref
OriginalExperiment = _ref.Experiment
bl.Experiment = FakeExperiment
_ref.Experiment = FakeExperiment  # patch where performLocalizationCluster resolves it

try:
    fake_full_experiment = FakeExperiment()
    fake_reference = FakeExperiment(cluster_locs=np.array([0.3, -0.2])).generate_recruitment_curves(
        amp_lims=[0, 1],
        n_steps=10,
    ).recruitment_values

    small_grid = create_loc_candidates(
        nerve_radius=1.5,
        limCandidateStd=(0.2, 0.3),
        limCandidateNum=(10, 20),
        nTriesLocs=(6, 6),
        nTriesStd=2,
        nTriesNum=2,
    )

    scaler = StandardScaler()
    small_grid_std = scaler.fit_transform(small_grid)

    experiment_info = {
        "full_experiment": fake_full_experiment,
        "full_lfm": fake_full_experiment.lead_field_matrix,
        "amp_lims": np.array([0, 1]),
        "n_stims_per_site": 10,
    }

    kernel = Matern(
        length_scale=[1.0, 1.0, 1.0, 1.0],
        nu=2.5,
        length_scale_bounds=(1e-5, float("inf")),
    )

    result = localize_functional_cluster(
        experiment_info=experiment_info,
        reference_curves=fake_reference,
        candidates_grid=small_grid,
        candidates_grid_standardized=small_grid_std,
        kernel=kernel,
        tradeoff=0.01,
        max_iter=3,
        batch_size=1,
        initial_random_samples=5,
        random_state=0,
    )

    print("Best score:", result.best_score)
    print("Number of iterations:", len(result.y_iter))
    assert len(result.y_iter) > 0
    print("Mini localization software test OK")

finally:
    bl.Experiment = OriginalExperiment
    _ref.Experiment = OriginalExperiment

2
4.9583191801728096e-11
-0.002550413854041312
Best score: -0.002550413854041312
Number of iterations: 1
Mini localization software test OK


## 5. Test with real data

In [ ]:
# from pathlib import Path
# import pickle
#
# base_path = Path(r"C:/Users/laura/OneDrive/Documenti/GitHub/INFORM/data/...")
#
# with open(base_path / "experiment_trial_cross.pkl", "rb") as f:
#     full_experiment = pickle.load(f)
#
# rc = full_experiment.generate_recruitment_curves(
#     amp_lims=[0, 1],
#     n_steps=20,
#     method="from_self",
# )
#
# print(rc.recruitment_values.shape)
